# Classification with Logistic Regression

## What you'll learn
- How classification predicts a category (here: will an account pay late?)
- Logistic regression, and why its features should be scaled
- Predicted labels vs predicted probabilities
- Accuracy, and why it can mislead
- Moving the decision threshold to catch more late payers

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

In [ ]:
from sklearn.model_selection import train_test_split

X = df[NUMERIC]
y = df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Late payers: {y.mean():.1%} overall, {y_test.mean():.1%} in the test set")

`stratify=y` keeps the same share of late payers in the train and test sets, which matters
when one class is much rarer than the other.

## Logistic regression

Despite the name, **logistic regression** is a classification model. It computes a
weighted sum of the features (like linear regression) and squashes it into a probability
between 0 and 1. Above 0.5 means "late".

It works best when features are on similar scales (days to pay is in tens, spend is in
hundreds of thousands), so we put a `StandardScaler` in front of it. `make_pipeline` chains
the two so they are fitted and used together; lesson 6 covers pipelines in depth.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

predicted = model.predict(X_test)          # 0 or 1
probability = model.predict_proba(X_test)[:, 1]   # chance of paying late
print(predicted[:10])
print(probability[:10].round(2))

## Accuracy, and its trap

**Accuracy** is the share of predictions that are right. But fewer than one in five
accounts pay late, so a model that always says "on time" is already about 82% accurate
while being useless. Always compare with that baseline.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score

always_on_time = [0] * len(y_test)
print(f"Always 'on time' accuracy: {accuracy_score(y_test, always_on_time):.1%}")
print(f"Logistic regression:       {accuracy_score(y_test, predicted):.1%}")
print(f"Late payers caught (recall): {recall_score(y_test, predicted):.1%}")

**Recall** is the share of real late payers the model caught. For a collections team,
missing a late payer usually costs more than a false alarm.

## Moving the threshold

The model says "late" when the probability is above 0.5. Lowering the threshold catches
more late payers at the cost of more false alarms. The right threshold is a business
decision, not a technical one.

In [ ]:
for threshold in [0.5, 0.4, 0.3]:
    flagged = (probability >= threshold).astype(int)
    print(f"threshold {threshold}: flags {flagged.sum():3d} accounts, "
          f"catches {recall_score(y_test, flagged):.0%} of late payers, "
          f"accuracy {accuracy_score(y_test, flagged):.1%}")

## What the model learned

With scaled features, the coefficients can be compared: larger positive values push
towards "late", negative values towards "on time".

In [ ]:
coefs = pd.Series(model[-1].coef_[0], index=NUMERIC).sort_values()
print(coefs.round(2))

## Practice

1. Train the model without the scaler (`LogisticRegression(max_iter=5000)` on its own).
   Is the accuracy different?
2. Which threshold flags about 30% of the test accounts?
3. Find the five test accounts with the highest late-payment probability.
4. Remove `avg_days_to_pay` from the features. How much does recall drop at 0.5?

In [ ]:
# Your practice code here

## Summary

- Classification predicts a category; logistic regression outputs a probability.
- Scale features for logistic regression; a pipeline keeps scaler and model together.
- `predict` gives labels; `predict_proba` gives probabilities.
- Accuracy can mislead when classes are unbalanced; check recall and the baseline.
- The decision threshold is a business choice between catching more and false alarms.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

X, y = df[NUMERIC], df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
probability = model.predict_proba(X_test)[:, 1]

# 1.
plain = LogisticRegression(max_iter=5000).fit(X_train, y_train)
print(f"without scaler: {accuracy_score(y_test, plain.predict(X_test)):.1%}, "
      f"with scaler: {accuracy_score(y_test, model.predict(X_test)):.1%}")

# 2.
for threshold in [0.5, 0.4, 0.35, 0.3, 0.25]:
    print(threshold, f"{(probability >= threshold).mean():.0%} flagged")

# 3.
top = X_test.assign(probability=probability).sort_values("probability", ascending=False)
print(df.loc[top.index[:5], ["account_id", "segment", "avg_days_to_pay"]].assign(
    probability=top["probability"].head(5).round(2)))

# 4.
fewer = [c for c in NUMERIC if c != "avg_days_to_pay"]
m = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train[fewer], y_train)
print(f"recall without days to pay: {recall_score(y_test, m.predict(X_test[fewer])):.1%} "
      f"vs {recall_score(y_test, model.predict(X_test)):.1%}")